# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irssaa29/Machine-learning_01/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: *Decision Tree.* My Week 1 experiments (on the starter CSV) showed that a shallow, readable tree could match or approach a hand-written rule's performance, and my Week 4 baseline itself is fundamentally a readable rule. A Decision Tree keeps that same readability — I can print it and explain exactly which conditions drive a "review" recommendation — while still letting the model discover feature thresholds and interactions I didn't hand-pick, unlike the baseline's fixed CTR-gap formula. Per this week's method-selection guide, my lane is a ranking/scoring question ("which pages first?"), so I'll use the tree's predicted probability, evaluated at Precision@K — not a plain yes/no classification — to stay consistent with how I've evaluated every rule so far.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [9]:
import os, sys, subprocess
import pandas as pd
import numpy as np

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/irssaa29/Machine-learning_01"
REPO_DIR = "Machine-learning_01"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "scikit-learn"], check=True)

from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

# --- Rebuild df_pair exactly as in Week 3/4, but keep client_hash_id this time ---
df_march = pd.read_parquet(
    "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/"
)
df_march["report_date"] = pd.to_datetime(df_march["report_date"])
first_half = df_march[df_march["report_date"].dt.day <= 15]
second_half = df_march[df_march["report_date"].dt.day > 15]

feat = first_half.groupby(["content_hash_id", "client_hash_id"]).agg(
    h1_impressions=("gsc_impressions", "sum"),
    h1_avg_position=("gsc_avg_position", "mean"),
    h1_clicks=("gsc_clicks", "sum"),
    h1_active_days=("report_date", "nunique"),
).reset_index()
feat["h1_ctr"] = feat["h1_clicks"] / feat["h1_impressions"].replace(0, pd.NA)

h2 = second_half.groupby("content_hash_id").agg(h2_clicks=("gsc_clicks", "sum")).reset_index()
df_pair = feat.merge(h2, on="content_hash_id", how="inner")
df_pair["is_declining"] = (df_pair["h2_clicks"] < df_pair["h1_clicks"]).astype(int)

# Match Week 4: drop no_data / no-position pages
df_pair = df_pair[(df_pair["h1_avg_position"].notna()) & (df_pair["h1_avg_position"] > 0)].copy()

print("Total valid pages:", len(df_pair))
print("Unique clients:", df_pair["client_hash_id"].nunique())

# --- Client-grouped split: a client's pages NEVER appear in both train and test ---
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df_pair, groups=df_pair["client_hash_id"]))

df_train = df_pair.iloc[train_idx].copy()
df_test = df_pair.iloc[test_idx].copy()

print("Train pages:", len(df_train), " | Train clients:", df_train["client_hash_id"].nunique())
print("Test pages:", len(df_test), " | Test clients:", df_test["client_hash_id"].nunique())

# Confirm zero client overlap
overlap = set(df_train["client_hash_id"]) & set(df_test["client_hash_id"])
print("Clients appearing in BOTH train and test (should be 0):", len(overlap))


Total valid pages: 150674
Unique clients: 44
Train pages: 109395  | Train clients: 30
Test pages: 41279  | Test clients: 14
Clients appearing in BOTH train and test (should be 0): 0


I used a client-grouped 70/30 split (GroupShuffleSplit), ensuring no client's pages appear in both train and test. This matters because a model could otherwise learn client-specific quirks (a particular client's naming conventions, industry, or baseline traffic level) rather than genuinely general signals about content decline — which would make my evaluation dishonestly optimistic, the same in-sample trap from Week 1's "leaky tree" lesson, just at the client level instead of the feature level.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I'll train a Decision Tree on the same 5 features as my Week 4 baseline, fit only on df_train, then evaluate Precision@20 and Precision@50 on df_test — the held-out clients the model never saw. I'll compute my Week 4 baseline's score on this same df_test slice too, so the comparison is apples-to-apples: same data, same split, same metric.

In [10]:
from sklearn.tree import DecisionTreeClassifier, export_text

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

features = ["h1_impressions", "h1_avg_position", "h1_clicks", "h1_ctr", "h1_active_days"]

X_train = df_train[features].fillna(0)
y_train = df_train["is_declining"].values
X_test = df_test[features].fillna(0)
y_test = df_test["is_declining"].values

# --- Train the tree ---
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree.fit(X_train, y_train)
tree_scores = tree.predict_proba(X_test)[:, 1]

# --- Recompute the Week 4 baseline score, but on this test set only ---
tier_benchmark = df_train.groupby(
    pd.cut(df_train["h1_avg_position"], bins=[0, 10, 30, 1000], labels=["good", "mid", "poor"])
)["h1_ctr"].mean()

def baseline_score_row(row):
    tier = pd.cut([row["h1_avg_position"]], bins=[0, 10, 30, 1000], labels=["good", "mid", "poor"])[0]
    benchmark = tier_benchmark.get(tier, 0)
    gap = max(benchmark - row["h1_ctr"], 0)
    return gap * row["h1_impressions"] if row["h1_impressions"] >= 100 else 0

df_test["baseline_score"] = df_test.apply(baseline_score_row, axis=1)

# --- Comparison table ---
base_rate = y_test.mean()
print(f"Base rate (Test set declining share): {base_rate:.3f}\n")

results = []
for k in (20, 50):
    b = precision_at_k(df_test["baseline_score"].values, y_test, k)
    t = precision_at_k(tree_scores, y_test, k)
    results.append({"k": k, "baseline_precision": round(b, 3), "tree_precision": round(t, 3)})

results_df = pd.DataFrame(results)
print(results_df)

print("\n--- Tree structure (readable) ---")
print(export_text(tree, feature_names=features))


/tmp/ipykernel_2091/4072875880.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = df_train[features].fillna(0)
/tmp/ipykernel_2091/4072875880.py:12: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test = df_test[features].fillna(0)
/tmp/ipykernel_2091/4072875880.py:21: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_benchmark = df_train.groupby(


Base rate (Test set declining share): 0.233

    k  baseline_precision  tree_precision
0  20                0.35             0.7
1  50                0.42             0.8

--- Tree structure (readable) ---
|--- h1_ctr <= 0.00
|   |--- class: 0
|--- h1_ctr >  0.00
|   |--- h1_ctr <= 0.00
|   |   |--- h1_ctr <= 0.00
|   |   |   |--- class: 1
|   |   |--- h1_ctr >  0.00
|   |   |   |--- class: 1
|   |--- h1_ctr >  0.00
|   |   |--- h1_impressions <= 75.50
|   |   |   |--- class: 1
|   |   |--- h1_impressions >  75.50
|   |   |   |--- class: 1



In [11]:
# Feature importances — what does the tree actually lean on?
importances = pd.Series(tree.feature_importances_, index=features).sort_values(ascending=False)
print("Feature importances:\n", importances)

# Sanity check the structural explanation: does h1_clicks==0 fully explain the first split?
zero_ctr_pages = df_train[df_train["h1_ctr"] == 0]
print("\nPages with h1_ctr == 0:", len(zero_ctr_pages))
print("Of those, h1_clicks == 0:", (zero_ctr_pages["h1_clicks"] == 0).sum())
print("Declining rate among h1_ctr == 0 pages:", zero_ctr_pages["is_declining"].mean())

# Class balance among ctr > 0 pages, to check the "almost everything predicts declining" pattern
nonzero_ctr_pages = df_train[df_train["h1_ctr"] > 0]
print("\nDeclining rate among h1_ctr > 0 pages:", nonzero_ctr_pages["is_declining"].mean())

Feature importances:
 h1_ctr             0.998781
h1_impressions     0.001219
h1_avg_position    0.000000
h1_clicks          0.000000
h1_active_days     0.000000
dtype: float64

Pages with h1_ctr == 0: 74057
Of those, h1_clicks == 0: 74057
Declining rate among h1_ctr == 0 pages: 0.0

Declining rate among h1_ctr > 0 pages: 0.5463806667043976


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the model leans on:**h1_ctr accounts for 99.9% of the tree's decisions (feature importance), with h1_impressions contributing 0.1% and the remaining three features unused entirely. This is not a sign of a well-generalizing model — it's a structural artifact of my label definition. Every page with h1_ctr == 0 also has h1_clicks == 0 (confirmed: 74,057/74,057 pages), and since is_declining = h2_clicks < h1_clicks, a page with zero first-half clicks can never mathematically decline (confirmed: 0.0% declining rate in this group). The tree's first split is essentially rediscovering this mathematical floor, not learning genuine content signal.

What this means for my headline numbers: my reported Precision@20 (0.70) and Precision@50 (0.80), while technically computed correctly on held-out clients, are inflated by this structural artifact. Once I restrict to pages that could plausibly decline at all (h1_ctr > 0), the underlying rate is close to a coin flip (54.6%), meaning the tree has very little real signal to work with beyond "did this page get clicked at all."

Honest next step: a fairer comparison would restrict both the baseline and the model to only pages with h1_ctr > 0 (pages that could genuinely decline), removing thezero-click floor from both sides equally. I'll do that check below before finalizing this comparison.

**Three concrete wrong cases:** (to be filled after re-running restricted to ctr > 0 — see below)

In [12]:
# Restrict train/test to pages with h1_ctr > 0, where decline is actually possible
df_train_restricted = df_train[df_train["h1_ctr"] > 0].copy()
df_test_restricted = df_test[df_test["h1_ctr"] > 0].copy()

X_train_r = df_train_restricted[features].fillna(0)
y_train_r = df_train_restricted["is_declining"].values
X_test_r = df_test_restricted[features].fillna(0)
y_test_r = df_test_restricted["is_declining"].values

tree_r = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42)
tree_r.fit(X_train_r, y_train_r)
tree_scores_r = tree_r.predict_proba(X_test_r)[:, 1]

base_rate_r = y_test_r.mean()
print(f"Restricted base rate (ctr>0 only): {base_rate_r:.3f}\n")

for k in (20, 50):
    t = precision_at_k(tree_scores_r, y_test_r, k)
    print(f"Restricted Precision@{k}: {t:.3f}")

importances_r = pd.Series(tree_r.feature_importances_, index=features).sort_values(ascending=False)
print("\nFeature importances (restricted):\n", importances_r)


Restricted base rate (ctr>0 only): 0.583

Restricted Precision@20: 0.900
Restricted Precision@50: 0.840

Feature importances (restricted):
 h1_ctr             0.442371
h1_avg_position    0.314929
h1_active_days     0.135805
h1_impressions     0.106895
h1_clicks          0.000000
dtype: float64


/tmp/ipykernel_2091/31596506.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train_r = df_train_restricted[features].fillna(0)
/tmp/ipykernel_2091/31596506.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test_r = df_test_restricted[features].fillna(0)


In [13]:
df_test_restricted = df_test_restricted.reset_index(drop=True)
df_test_restricted["tree_score"] = tree_scores_r
df_test_restricted["predicted_declining"] = (tree_scores_r > 0.5).astype(int)

wrong_cases = df_test_restricted[df_test_restricted["predicted_declining"] != df_test_restricted["is_declining"]]
print("Total wrong cases:", len(wrong_cases))

sample_wrong = wrong_cases.sample(3, random_state=42)[["content_hash_id", "h1_ctr", "h1_avg_position", "h1_active_days", "h1_impressions", "tree_score", "is_declining"]]
print(sample_wrong.to_string(index=False))

Total wrong cases: 7457
         content_hash_id    h1_ctr  h1_avg_position  h1_active_days  h1_impressions  tree_score  is_declining
content_c2ed3b73c9024928  0.011111         2.037063              13             180    0.230965             1
content_4a7d4cd718097a77   0.00395         3.435150              15            1519    0.431026             1
content_28a6dcd34fdf6f0f  0.002833         5.012791              15            1059    0.431026             1


Three concrete wrong cases (all false negatives — predicted stable, actually declined):

1. **content_c2ed3b73c9024928** — position 2.0 (excellent), 180 impressions, active 13/15 days, tree_score 0.23 (confidently "not declining"), but actually declined.
2. **content_4a7d4cd718097a77** — position 3.4, 1,519 impressions (highest of the three), active all 15 days, tree_score 0.43, but actually declined.
3. **content_28a6dcd34fdf6f0f** — position 5.0, 1,059 impressions, active 15/15 days, tree_score 0.43, but actually declined.

**Why these are hard:** all three look genuinely healthy by every first-half signal available — strong position, consistent visibility, real traffic. The model reasonably treated strong current performance as a sign of stability. But this reveals a real limit of my feature set: it captures a page's state in the first half of March, not the forces acting on it (a competitor overtaking it, a seasonal shift, a new SERP feature). A page can be doing everything right and still decline for reasons invisible to these features. This is a legitimate limitation to report, not a fixable bug — it points toward needing trend-over-time features (e.g., position trajectory across several months) rather than a single-period snapshot, for future work.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.